In [4]:
import sec_parser as sp

from sec_parser.semantic_elements import (
    TextElement,
    TitleElement,
    TableElement,
    TopSectionTitle,
    SupplementaryText,
)

parser = sp.Edgar10KParser()

In [5]:
def parse_10k_file(path):
    html = path.read_text(encoding="utf-8", errors='ignore')
    elements = parser.parse(html)

    chunks = []
    section_title = None
    ticker = path.stem

    for order, element in enumerate(elements):
        if isinstance(element, (TopSectionTitle, TitleElement)):
            section_title = element.text.strip()
            continue

        elif isinstance(element, TextElement):
            text = element.text.strip()
            chunk_type = 'text'
        elif isinstance(element, TableElement):
            text = element.table_to_markdown().strip()
            chunk_type = 'table'

        elif isinstance(element, SupplementaryText):
            text = element.text.strip()
            chunk_type = 'supplementary_text'
        
        else: 
            continue
        
        if not text:
            continue
        
        chunks.append({
            "ticker": ticker,
            "order": order,
            "chunk_type": chunk_type,
            "section_title": section_title,
            "text": text,
        })

    return chunks



In [6]:
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

embedding_model_name = "BAAI/bge-small-en-v1.5"

tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=400,
    chunk_overlap=50,
)

/home/wagyu0923/miniconda3/envs/nlp/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
def count_tokens(text):
    return len(tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"])

In [8]:
def clean_section_title(section_title, max_tokens=32):
    if not section_title:
        return None

    section_title = section_title.strip()

    if count_tokens(section_title) > max_tokens:
        return None

    return section_title

def build_embedding_text(section_title, text):
    if section_title:
        return f"{section_title}\n\n{text}"
    
    return text

def split_row(row):
    parts = text_splitter.split_text(row["text"])
    section_title = clean_section_title(row["section_title"])
    records=[]

    for part in parts:
        records.append({
            "ticker": row["ticker"],
            "chunk_type": row["chunk_type"],
            "section_title": section_title,
            "text": part,
            "embedding_text": build_embedding_text(section_title, part)
        })

    return records


In [9]:
from tqdm.auto import tqdm
import warnings
from pathlib import Path
import pandas as pd

corpus_dir = Path("../10k")
html_files = sorted(corpus_dir.glob("*.html"))
len(html_files),html_files[:5]
all_chunks = []
failed_files = []

for path in tqdm(html_files):
    try:
        with warnings.catch_warnings():
            warnings.filterwarnings(
                "ignore",
                message="Invalid section type.*",
                category=UserWarning,
            )
            file_chunks = parse_10k_file(path)
            
        all_chunks.extend(file_chunks)
    except Exception as e:
        failed_files.append({
            "file": path.name,
            "error": repr(e),
        })

all_corpus_df = pd.DataFrame(all_chunks)

all_corpus_df.shape, len(failed_files)

  9%|▉         | 47/497 [01:44<12:43,  1.70s/it]/home/wagyu0923/miniconda3/envs/nlp/lib/python3.11/site-packages/sec_parser/utils/bs4_/approx_table_metrics.py:31: UserWarning: Failed to get table metrics: 'NoneType' object has no attribute 'text'
  warnings.warn(f"Failed to get table metrics: {e}", stacklevel=0)
 10%|█         | 52/497 [01:56<23:08,  3.12s/it]/home/wagyu0923/miniconda3/envs/nlp/lib/python3.11/site-packages/sec_parser/utils/bs4_/approx_table_metrics.py:31: UserWarning: Failed to get table metrics: 'NoneType' object has no attribute 'text'
  warnings.warn(f"Failed to get table metrics: {e}", stacklevel=0)
 28%|██▊       | 140/497 [04:39<09:36,  1.61s/it]/home/wagyu0923/miniconda3/envs/nlp/lib/python3.11/site-packages/sec_parser/utils/bs4_/approx_table_metrics.py:31: UserWarning: Failed to get table metrics: 'NoneType' object has no attribute 'text'
  warnings.warn(f"Failed to get table metrics: {e}", stacklevel=0)
 31%|███       | 154/497 [05:10<10:51,  1.90s/it]/home/wa

((223544, 5), 0)

In [10]:
all_corpus_df = all_corpus_df[all_corpus_df["section_title"].notna()].reset_index(drop=True)

In [11]:
all_corpus_df.to_parquet("all_corpus_structural_chunks.parquet", index=False)

In [12]:
all_final_rows = []

for _, row in tqdm(all_corpus_df.iterrows(), total=len(all_corpus_df)):
    all_final_rows.extend(split_row(row))

all_final_df = pd.DataFrame(all_final_rows)

all_final_df = all_final_df.reset_index(drop=True)
all_final_df["chunk_id"] = all_final_df.index

all_final_df = all_final_df[
    ["chunk_id", "ticker", "chunk_type", "section_title", "text", "embedding_text"]
]

all_final_df.shape, all_final_df.head()

100%|██████████| 223479/223479 [11:52<00:00, 313.56it/s]


((301184, 6),
    chunk_id ticker chunk_type                               section_title  \
 0         0      A       text                                    Overview   
 1         1      A       text                                    Overview   
 2         2      A       text  Life Sciences and Applied Markets Business   
 3         3      A       text           Life Sciences and Applied Markets   
 4         4      A       text           Life Sciences and Applied Markets   
 
                                                 text  \
 0  Agilent Technologies Inc. ("we", "Agilent" or ...   
 1  and life sciences research areas to interrogat...   
 2  Our life sciences and applied markets business...   
 3  The Pharmaceutical, Biopharmaceutical, CRO & C...   
 4  Our products are used to test for safety, qual...   
 
                                       embedding_text  
 0  Overview\n\nAgilent Technologies Inc. ("we", "...  
 1  Overview\n\nand life sciences research areas t...  
 2  

In [13]:
embedding_token_counts = all_final_df["embedding_text"].apply(count_tokens)


In [14]:
embedding_token_counts.describe(), (embedding_token_counts>512).sum()

(count    301184.000000
 mean        208.717947
 std         142.419579
 min           2.000000
 25%          72.000000
 50%         188.000000
 75%         366.000000
 max         432.000000
 Name: embedding_text, dtype: float64,
 np.int64(0))

In [16]:
all_final_df.to_parquet("all_corpus_final_chunks.parquet", index=False)